# **Histogram Eşitleme ile Gürültü Ekleme ve Kaldırma ve Kontrastı Düzeltme**

####**Bu derste şunları öğreneceğiz:**
1. Görüntülere beyaz gürültü veya film greni efektleri ekleme
2. Histogram Eşitleme nasıl uygulanır

### **Gürültü Nedir?**

<img src="ISO-Noise.jpg" width="500">

Dijital Kamera sensörleri, kamera sensörünün (CCD) hassasiyetini artırarak düşük ışıklı ortamlarda fotoğraf çekebilir. Ancak, hassasiyetteki bu artışın (ISO artışı) bedeli gürültüdür. Gürültü, sensörün yüksek duyarlılığının onu rastgele gürültüye duyarlı hale getirmesi nedeniyle ortaya çıkar. Bunun nedeni, düşük ışıklı sahnelerde sahne ile rastgele foton gürültüsü arasında çok fazla değişiklik olmamasıdır. 

https://blog.michaeldanielho.com/2016/08/understanding-cameras-exposure-setting.html

In [ ]:

import cv2
import random
import numpy as np
from matplotlib import pyplot as plt

 
def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()


## **Görüntülere Gürültü Ekleme**
Bu kod, temel olarak bir görüntüye gürültü ekleyerek kalitesini düşürür veya bozarak belirli algoritmaların performansını test etmek için bir simülasyon aracı olarak kullanılabilir. Örneğin, bir gürültü azaltma (denoising) algoritması geliştiriliyorsa, önce bu kodla yapay gürültü eklenir, ardından geliştirilen algoritmanın bu gürültüyü ne kadar etkili bir şekilde temizlediği test edilir.

In [ ]:
def addWhiteNoise(image):  # Bu fonksiyon, bir görüntüyü (image) girdi olarak alır ve gürültü eklenmiş halini döndürür. 
    # gürültünün ne kadar yoğun olacağını belirleyen rastgele bir olasılık değeri oluşturur. 
    # 
    #Bu, piksellerin %5 ile %10'unun gürültüden etkileneceği anlamına gelir.
    prob = random.uniform(0.05, 0.1) # Değer 0.05 ile 0.1 arasında ondalıklı bir sayıdır. 

    
    rnd = np.random.rand(image.shape[0], image.shape[1]) 
    # görüntünün boyutuyla aynı boyutta, 0 ile 1 arasında rastgele sayılar içeren bir matris (array) oluşturur. 
    # Bu matris, hangi piksellerin gürültüden etkileneceğini belirlemek için kullanılır.
    
    # Bu satırda, az önce oluşturulan rnd matrisindeki değerlerden, prob değişkeninden daha küçük olanlar bulunur. 
    # Bu koşulu sağlayan piksellerin, RGB değerleri (yani parlaklığı), 
    # 50 ile 230 arasında rastgele seçilen yeni bir tam sayıya atanır.
    # 50 ile 230 arasındaki değerler, pikselin ne tam siyah (0) ne de tam beyaz (255) olmasını sağlar. 
    # Bu sayede, "beyaz gürültü" olarak adlandırılsa da, piksel değerleri gri tonlarında kalır 
    # ve daha doğal bir gürültü efekti yaratır.
    image[rnd < prob] = np.random.randint(50,230)
    return image

In [ ]:
# Resmimizi yükleyin
image = cv2.imread('../files/images/america35.jpg')
imshow("Input Image", image,6)

# Beyaz gürültü fonksiyonumuzu girdi görüntümüze uygulayın 
noise_1 = addWhiteNoise(image)
imshow("Noise Added", noise_1)

**Yerel Olmayan Araçlar Denoising'in 4 çeşidi vardır:**

- cv2.fastNlMeansDenoising() - tek bir gri tonlamalı görüntü ile çalışır
- cv2.fastNlMeansDenoisingColored() - renkli bir görüntü ile çalışır.
- cv2.fastNlMeansDenoisingMulti() - Bu fonksiyon, bir video gibi, kısa bir süre içinde çekilmiş, gürültü içeren bir dizi gri tonlamalı resmi işlemek için idealdir. Arka arkaya gelen karelerdeki benzerlikleri kullanarak daha etkili bir gürültü giderme sağlar.
- cv2.fastNlMeansDenoisingColoredMulti() - yukarıdaki ile aynı, ancak renkli görüntüler için.


In [ ]:
# cv2.fastNlMeansDenoisingColored(src, dst, h, hColor, templateWindowSize, searchWindowSize)
# src: Gürültü içeren giriş resmidir. Bu, 8 bitlik, 3 kanallı (renkli) bir resim olmalıdır.
# dst: İşlemin sonucunda gürültüden arındırılmış resmin kaydedileceği çıkış resmidir. 
    # Genellikle None olarak ayarlanır, bu durumda fonksiyon yeni bir resim oluşturur ve onu döndürür.
# h: Gürültü giderme gücünü kontrol eden en önemli parametredir. 
    # Bu değer ne kadar büyük olursa, resimdeki gürültü o kadar çok giderilir. 
    # Ancak, çok yüksek bir değer, resimdeki önemli detayların da kaybolmasına ve bulanıklaşmaya neden olabilir. 
    # Genellikle 3 ile 30 arasında bir değer kullanılır.
# hColor: h parametresine benzer şekilde çalışır, ancak resmin renk bileşenindeki gürültü giderme gücünü belirler. 
    # Genellikle h değerine yakın bir değer verilir.
# templateWindowSize: Ortalama alınacak örnek pencere boyutudur. 
    #Karşılaştırma için bir pikselin etrafında kullanılan kare pencereyi tanımlar. 
    # Değeri tek sayı olmalıdır. Varsayılan değeri 7'dir.
# searchWindowSize: Arama penceresi boyutudur. Benzer piksellerin aranacağı daha büyük pencereyi tanımlar. 
    # Değeri tek sayı olmalıdır. Varsayılan değeri 21'dir. 
    #Bu değer ne kadar büyük olursa, daha fazla benzer piksel bulunabilir, ancak işlem süresi de o kadar artar.

dst = cv2.fastNlMeansDenoisingColored(noise_1, None, 11, 6, 7, 21)

imshow("Noise Removed", dst)

### **Histogram Eşitleme Kullanma** 

<img src="histogram_equalization.jpg" width="500">

Bu, bir görüntünün dinamik aralığını 'ayarlayarak' yoğunluk dağılımını daha eşit bir şekilde yayılmasını sağlar ve böylece kontrastı iyileştirir.

#### **İlk olarak, Girdi Görüntümüzün Histogramına bir göz atalım**

In [ ]:
# Resmimizi yükleyin
img = cv2.imread('../files/images/america36.jpg')
imshow("Original", img) # Orijinal resim

gray_image = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  #  gri tonlamalı (grayscale) formata dönüştürme 

# Histogram dağılımımızı oluşturalım
hist,bins = np.histogram(gray_image.flatten(),256,[0,256]) 
# gray_image.flatten(): Görüntü, 2 boyutlu bir matrisken, histogram hesaplaması için 1 boyutlu bir diziye (piksel değerlerinin düz listesi) dönüştürülür.
# np.histogram(): Bu fonksiyon, bir dizinin histogramını hesaplar.
# 256: Histogramın bölüneceği grup (bin) sayısını belirtir. 
# Gri tonlamalı bir resimde 0'dan 255'e kadar 256 farklı parlaklık değeri olduğu için bu değer kullanılır.


# Kümülatif Toplamı Alalım. Örneğin [10, 5, 20, 15] dizisinin kümülatif toplamı [10, 15, 35, 50] olur. 
cdf = hist.cumsum()
# Kümülatif Toplam (cdf): cdf dizisindeki her bir eleman, o parlaklık seviyesine ve 
# ondan daha düşük olan tüm parlaklık seviyelerine sahip piksellerin toplam sayısını gösterir.

# Kümülatif dağılımı normalize edin
cdf_normalized = cdf * float(hist.max()) / cdf.max()
# cdf_normalized: CDF değerleri, görselleştirmenin daha iyi olması için 0-255 aralığına normalize edilir.

# CDF'mizi Histogramımızın üzerine çizin
plt.plot(cdf_normalized, color = 'b') # plt.plot(): Normalize edilmiş CDF'yi mavi bir çizgi olarak çizer.
plt.hist(gray_image.flatten(),256,[0,256], color = 'r') # plt.hist(): Orijinal histogramı kırmızı çubuklar halinde çizer.
plt.xlim([0,256]) # X ekseni aralığını 0 ile 256 olarak sınırlar.
plt.legend(('cdf','histogram'), loc = 'upper left') # Grafik üzerindeki mavi çizginin cdf, kırmızı çubukların ise histogram olduğunu belirtir.
plt.show() #Oluşturulan grafiği bir pencerede görüntüler.
imshow("gray_image", gray_image)


#### **Şimdi Histogram Eşitleme uygulayalım**

In [ ]:
img = cv2.imread('../files/images/america36.jpg')

# Gri tonlamaya dönüştür
gray_image = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
imshow("gray_image", gray_image)
# Histogramı eşitleyelim
gray_image = cv2.equalizeHist(gray_image)
imshow("equalizeHist", gray_image)

# Histogram dağılımımızı oluşturun
hist,bins = np.histogram(gray_image.flatten(),256,[0,256])

# Kümülatif Toplamı Alın 
cdf = hist.cumsum()

# Kümülatif dağılımı normalize edin
cdf_normalized = cdf * float(hist.max()) / cdf.max()

# CDF'mizi Histogramımızın üzerine çizin
plt.plot(cdf_normalized, color = 'b')
plt.hist(gray_image.flatten(),256,[0,256], color = 'r')
plt.xlim([0,256])
plt.legend(('cdf','histogram'), loc = 'upper left')
plt.show()

Bu görüntünün tüm RGB (BGR) kanallarını eşitleyelim ve ardından eşitlenmiş bir renkli görüntü elde etmek için bunları birleştirelim

In [ ]:
import cv2 
 
img = cv2.imread('../files/images/america36.jpg')
 
imshow("Original", img)
 
# Histogramımızı eşitleyin
# Varsayılan renk biçimi BGR'dır
 
red_channel = img[:, :, 2]
red = cv2.equalizeHist(red_channel)
 
green_channel = img[:, :, 1]
green = cv2.equalizeHist(green_channel)
 
blue_channel = img[:, :, 0]
blue = cv2.equalizeHist(blue_channel)
 
# src görüntüsüyle aynı şekle sahip boş görüntü oluşturalım
red_img = np.zeros(img.shape)
red_img[:,:,2] = red
red_img = np.array(red_img, dtype=np.uint8)
imshow("Red", red_img)
 
green_img = np.zeros(img.shape)
green_img[:,:,1] = green
green_img = np.array(green_img, dtype=np.uint8)
imshow("Green", green_img)
 
blue_img = np.zeros(img.shape)
blue_img[:,:,0] = blue
blue_img = np.array(blue_img, dtype=np.uint8)
imshow("Blue", blue_img)
 
merged = cv2.merge([blue, green, red])
imshow("Merged", merged)